In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.20
MISSING_RATE = 0.10
NOISE_STD    = 5.0
POLY_DEGREES = [1, 2, 3]
ALPHA_GRID   = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
CV_FOLDS     = 5
COLUMNS = ['CRIM','ZN','INDUS','CHAS','NOX','RM','AGE',
            'DIS','RAD','TAX','PTRATIO','B','LSTAT','MEDV']

np.random.seed(RANDOM_STATE)

df = pd.read_csv('housing.csv', sep=r'\s+', header=None, names=COLUMNS)
X_orig, y_orig = df.drop(columns=['MEDV']), df['MEDV']
def split(X, y):
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

def make_result(label, model, X_train, X_test, y_train, y_test, **extra):
    tr_pred, te_pred = model.predict(X_train), model.predict(X_test)
    return {'Config': label, **extra,
            'Train MSE': round(mean_squared_error(y_train, tr_pred), 3),
            'Test MSE':  round(mean_squared_error(y_test,  te_pred), 3),
            'Train R²':  round(r2_score(y_train, tr_pred), 3),
            'Test R²':   round(r2_score(y_test,  te_pred), 3)}

def fit_model(estimator, param_grid, X_train, y_train):
    if param_grid:
        gs = GridSearchCV(estimator, param_grid, cv=CV_FOLDS)
        gs.fit(X_train, y_train)
        return gs.best_estimator_, gs.best_params_
    return estimator.fit(X_train, y_train), {}

print("=" * 60)
print(" PROBLEM 1: HANDLING MISSING DATA")
print("=" * 60)

X_missing = X_orig.copy()
n_missing = int(MISSING_RATE * len(X_missing))
for col in ['CRIM', 'TAX', 'RM']:
    X_missing.loc[np.random.choice(len(X_missing), n_missing, replace=False), col] = np.nan

X_median = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_missing), columns=X_orig.columns)
X_interp = X_missing.interpolate(method='linear', limit_direction='both')

rows_p1 = []
for name, X in [('Original', X_orig), ('Imputed (Median)', X_median), ('Imputed (Interpolation)', X_interp)]:
    Xtr, Xte, ytr, yte = split(X, y_orig)
    rows_p1.append(make_result(name, LinearRegression().fit(Xtr, ytr), Xtr, Xte, ytr, yte))

display(pd.DataFrame(rows_p1))

# ── Problem 2 ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" PROBLEM 2: REGRESSION ON NOISY DATA")
print("=" * 60)

y_noisy = y_orig + np.random.normal(0, NOISE_STD, size=len(y_orig))
Xtr, Xte, ytr_clean, yte_clean = split(X_orig, y_orig)
_,   _,   ytr_noisy, yte_noisy = split(X_orig, y_noisy)

def run_experiment(Xtr, Xte, ytr, yte, title):
    rows = []
    for d in POLY_DEGREES:
        for name, estimator, pgrid in [
            ('Linear', make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()), None),
            ('Ridge',  make_pipeline(PolynomialFeatures(d), StandardScaler(), Ridge()),            {'ridge__alpha': ALPHA_GRID}),
            ('Lasso',  make_pipeline(PolynomialFeatures(d), StandardScaler(), Lasso(max_iter=5000)), {'lasso__alpha': ALPHA_GRID}),
        ]:
            fitted, best = fit_model(estimator, pgrid, Xtr, ytr)
            rows.append(make_result(f'Deg {d} – {name}', fitted, Xtr, Xte, ytr, yte,
                                    **{'Best Alpha': best.get(f'{name.lower()}__alpha', '-')}))
    print(f'\n--- {title} ---')
    display(pd.DataFrame(rows))

run_experiment(Xtr, Xte, ytr_clean, yte_clean, 'WITHOUT NOISE')
run_experiment(Xtr, Xte, ytr_noisy, yte_noisy, 'WITH NOISE')

 PROBLEM 1: HANDLING MISSING DATA


,Config,Train MSE,Test MSE,Train R²,Test R²
0,Original,21.641,24.291,0.751,0.669
1,Imputed (Median),23.539,24.269,0.729,0.669
2,Imputed (Interpolation),22.688,23.417,0.739,0.681



 PROBLEM 2: REGRESSION ON NOISY DATA

--- WITHOUT NOISE ---


,Config,Best Alpha,Train MSE,Test MSE,Train R²,Test R²
0,Deg 1 – Linear,-,21.641,24.291,0.751,0.669
1,Deg 1 – Ridge,1.0,21.643,24.313,0.751,0.668
2,Deg 1 – Lasso,0.001,21.641,24.295,0.751,0.669
3,Deg 2 – Linear,-,5.131,14.257,0.941,0.806
4,Deg 2 – Ridge,1.0,6.978,11.207,0.920,0.847
5,Deg 2 – Lasso,0.01,7.313,12.027,0.916,0.836
6,Deg 3 – Linear,-,0.000,16476.011,1.000,-223.671
7,Deg 3 – Ridge,10.0,6.030,10.343,0.931,0.859
8,Deg 3 – Lasso,0.01,4.879,10.368,0.944,0.859



--- WITH NOISE ---


,Config,Best Alpha,Train MSE,Test MSE,Train R²,Test R²
0,Deg 1 – Linear,-,46.039,52.756,0.613,0.511
1,Deg 1 – Ridge,1.0,46.041,52.711,0.613,0.511
2,Deg 1 – Lasso,0.01,46.044,52.705,0.613,0.512
3,Deg 2 – Linear,-,23.969,51.515,0.798,0.523
4,Deg 2 – Ridge,10.0,33.659,38.728,0.717,0.641
5,Deg 2 – Lasso,0.1,37.980,39.526,0.681,0.634
6,Deg 3 – Linear,-,0.000,214162.929,1.000,-1983.935
7,Deg 3 – Ridge,100.0,33.252,38.418,0.720,0.644
8,Deg 3 – Lasso,0.1,34.591,37.511,0.709,0.652
